In [2]:
!pip install fastapi uvicorn joblib pydantic -q

In [3]:
# ============================================================
# Notebook 07: Model Persistence & FastAPI Inference
# AI Financial Risk Intelligence Platform
# ============================================================

from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# Load Dataset
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\user\machine-learning-models-practical\AIML Engineer Training Series\AI_Financial_Risk_Intelligence_Platform")

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "german.data"

column_names = [
    "status", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment_duration", "installment_rate", "personal_status_sex",
    "other_debtors", "present_residence", "property", "age",
    "other_installment_plans", "housing", "existing_credits", "job",
    "people_liable", "telephone", "foreign_worker", "target"
]

df = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    header=None,
    names=column_names
)

df["target"] = df["target"].map({1: 0, 2: 1})

# ------------------------------------------------------------
# Features and Target
# ------------------------------------------------------------

X = df.drop(columns="target")
y = df["target"]

# ------------------------------------------------------------
# Train/Test Split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ------------------------------------------------------------
# Feature Types
# ------------------------------------------------------------

categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
numerical_cols = X_train.select_dtypes(exclude="object").columns.tolist()

# ------------------------------------------------------------
# Preprocessor
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

# ------------------------------------------------------------
# Pipeline
# ------------------------------------------------------------

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

pipeline.fit(X_train, y_train)

print("Pipeline trained successfully")

Pipeline trained successfully


In [4]:
# ============================================================
# Save Trained Pipeline
# ============================================================

import joblib

MODEL_PATH = PROJECT_ROOT / "models" / "credit_risk_pipeline.joblib"

joblib.dump(pipeline, MODEL_PATH)

print("Model saved successfully")
print(MODEL_PATH)

Model saved successfully
C:\Users\user\machine-learning-models-practical\AIML Engineer Training Series\AI_Financial_Risk_Intelligence_Platform\models\credit_risk_pipeline.joblib


In [5]:
# ============================================================
# Load Saved Model
# ============================================================

loaded_pipeline = joblib.load(MODEL_PATH)

print("Model loaded successfully")

print(type(loaded_pipeline))

Model loaded successfully
<class 'sklearn.pipeline.Pipeline'>
